# 02 — Train the LoRA Adapter

Fine-tunes Qwen 2.5-7B-Instruct with QLoRA on the logistics Q&A dataset.

**Before you run:**
1. Switch Colab to a GPU runtime: Runtime → Change runtime type → T4 GPU
2. (Optional) Set `WANDB_API_KEY` in Colab secrets to enable Weights & Biases logging
3. Run `01_generate_dataset.ipynb` first so the train/val/test splits exist

**Runtime:** ~3-6 hours on a free T4 (16 GB VRAM).

**Recovery:** Checkpoints back up to Drive every 10 minutes during training. If
Colab disconnects, reopen the notebook in a fresh session and run all cells in
order — the restore cell pulls checkpoints back from Drive and `--resume` makes
training pick up where it stopped.

In [ ]:
# Confirm GPU
!nvidia-smi

## Setup

In [ ]:
!git clone https://github.com/masonsau0/logistics-qa-lora.git
%cd logistics-qa-lora
!pip install -q -r requirements.txt

In [ ]:
# Restore dataset from Google Drive
from google.colab import drive

drive.mount("/content/drive")
!cp /content/drive/MyDrive/logistics-qa-lora/data/*.jsonl data/
!ls -lh data/*.jsonl

## Restore checkpoints from Drive (if a previous run was interrupted)

If Colab disconnected during a prior training run, the latest checkpoints were
saved to Drive by the backup thread inside the training cell. This cell pulls
them back to the local VM so training can resume from where it stopped.

First-time run: this is a no-op.

In [ ]:
# Restore any prior checkpoints from Drive so --resume can pick them up
import os

drive_ckpts = "/content/drive/MyDrive/logistics-qa-lora/artifacts/checkpoints"
local_ckpts = "/content/logistics-qa-lora/artifacts/checkpoints"

if os.path.exists(drive_ckpts) and os.listdir(drive_ckpts):
    os.makedirs(local_ckpts, exist_ok=True)
    !cp -r {drive_ckpts}/* {local_ckpts}/
    print("Restored checkpoints from Drive:")
    !ls {local_ckpts}
else:
    print("No prior checkpoints on Drive — training will start from step 0")

In [ ]:
# Optional: W&B logging
import os

from google.colab import userdata

try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("W&B enabled")
    USE_WANDB = True
except Exception:
    print("WANDB_API_KEY not found in Colab secrets — training will run without W&B logging")
    USE_WANDB = False

## Train

Runs `src/train.py` with `--resume` so it picks up from the latest checkpoint
if one was restored above. A backup thread mirrors `artifacts/checkpoints/` to
Drive every 10 minutes — if Colab disconnects mid-training, you lose at most
10 min of work.

**Why one cell, not two:** Colab runs cells sequentially in the same kernel,
so a separate backup cell would queue and never run while training was active.
Threading inside this cell sidesteps that.

You'll see two kinds of output interleaved:
- Training logs: `{loss: ..., step: ...}` from the Trainer
- Backup logs every 10 min: `[backup] mirrored N checkpoint(s) → Drive at HH:MM:SS`

In [ ]:
import os
import subprocess
import threading
import time

src_ckpts = "/content/logistics-qa-lora/artifacts/checkpoints"
dst_ckpts = "/content/drive/MyDrive/logistics-qa-lora/artifacts/checkpoints"
os.makedirs(dst_ckpts, exist_ok=True)


def backup_loop():
    while True:
        try:
            if os.path.exists(src_ckpts) and os.listdir(src_ckpts):
                subprocess.run(
                    ["rsync", "-a", "--delete", f"{src_ckpts}/", f"{dst_ckpts}/"],
                    check=False,
                )
                n = len([d for d in os.listdir(src_ckpts) if d.startswith("checkpoint-")])
                print(
                    f"[backup] mirrored {n} checkpoint(s) → Drive at {time.strftime('%H:%M:%S')}",
                    flush=True,
                )
        except Exception as e:
            print(f"[backup error] {e}", flush=True)
        time.sleep(300)


threading.Thread(target=backup_loop, daemon=True).start()
print("Backup thread started — running in parallel with training below", flush=True)

# Tuned for Colab T4 free-tier with tight quota:
#   -u                    : unbuffered stdout so Trainer logs stream live
#   --epochs 1            : 1 pass through 5400 examples
#   --max-seq-length 512  : ~4x faster than 1024 (attention is O(n^2))
#   --save-steps 25       : checkpoint every ~12 min, so disconnects lose <15 min
cmd = [
    "python",
    "-u",
    "-m",
    "src.train",
    "--epochs",
    "1",
    "--batch-size",
    "4",
    "--grad-accum",
    "4",
    "--max-seq-length",
    "512",
    "--save-steps",
    "25",
    "--resume",
]
if not USE_WANDB:
    cmd.append("--no-wandb")
subprocess.run(cmd, check=False)
print("Training finished.", flush=True)

## Save the adapter to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/logistics-qa-lora/artifacts
!cp -r artifacts/checkpoints/final /content/drive/MyDrive/logistics-qa-lora/artifacts/
!ls -lh /content/drive/MyDrive/logistics-qa-lora/artifacts/final/

## Optional: push to Hugging Face Hub

Adapters are small (~80 MB) and the Hub is a nice place to host them. You'll need an HF token with `write` scope set as `HF_TOKEN` in Colab secrets.

In [ ]:
# Uncomment to push
# from huggingface_hub import HfApi, login
# login(token=userdata.get('HF_TOKEN'))
# api = HfApi()
# api.create_repo('YOUR_HF_USERNAME/qwen25-7b-logistics-lora', exist_ok=True)
# api.upload_folder(
#     folder_path='artifacts/checkpoints/final',
#     repo_id='YOUR_HF_USERNAME/qwen25-7b-logistics-lora',
# )

### Next step
Open `03_evaluate.ipynb` to measure how much the fine-tune helped.